# Xarray-Spatial Zonal Crosstab: Cross-tabulated spatial statistics

`zonal_crosstab` calculates cross-tabulated statistics between two classified raster datasets. Given a zones raster and a values raster, it counts or summarizes how value classes are distributed within each zone. This is useful for terrain characterization, land suitability assessment, and environmental analysis.

### What you'll build

1. Generate synthetic terrain and compute slope
2. Classify elevation and slope into discrete zones
3. Cross-tabulate slope distribution by elevation zone
4. Visualize the relationship as a stacked bar chart
5. Compare with zonal summary statistics

![Zonal crosstab preview](images/zonal_crosstab_preview.png)

**Jump to a section:**
[Terrain data](#Terrain-data) | [Slope](#Slope) | [Classification](#Classification) | [Crosstab](#Crosstab) | [Zonal statistics](#Zonal-statistics)

Standard imports plus `generate_terrain`, `slope`, `hillshade`, and zonal tools from xrspatial.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

import xrspatial

## Terrain data

Generate synthetic terrain using `generate_terrain`, which creates realistic elevation data from fractal noise. Hillshading adds a 3D effect for visualization.

In [ ]:
# Generate synthetic terrain
W, H = 800, 600
x_range = (-20e6, 20e6)
y_range = (-20e6, 20e6)

template = xr.DataArray(np.zeros((H, W)))
terrain = template.xrs.generate_terrain(x_range=x_range, y_range=y_range, seed=42, zfactor=4000)
terrain.name = "Elevation"

print(f"Terrain dimensions: {terrain.shape}")
print(f"Elevation range: {float(terrain.min()):.0f} - {float(terrain.max()):.0f} meters")

In [ ]:
# Create hillshade for visualization
illuminated = terrain.xrs.hillshade()

fig, ax = plt.subplots(figsize=(10, 7.5))
illuminated.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
terrain.plot.imshow(ax=ax, cmap='terrain', alpha=0.5, add_colorbar=True,
                    cbar_kwargs={'label': 'Elevation (m)'})
ax.set_title('Synthetic terrain')
ax.set_axis_off()
plt.tight_layout()

## Slope

Slope measures terrain steepness in degrees. Flat areas read 0, gentle slopes are under 15, and steep terrain exceeds 30.

In [ ]:
# Calculate slope in degrees
slope_agg = terrain.xrs.slope()
slope_agg.name = "Slope"

print(f"Slope range: {float(slope_agg.min()):.1f}° - {float(slope_agg.max()):.1f}°")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

terrain.plot.imshow(ax=axes[0], cmap='terrain', add_colorbar=True,
                    cbar_kwargs={'label': 'Elevation (m)'})
axes[0].set_title('Elevation')
axes[0].set_axis_off()

slope_agg.plot.imshow(ax=axes[1], cmap='YlOrRd', add_colorbar=True,
                      cbar_kwargs={'label': 'Slope (degrees)'})
axes[1].set_title('Slope')
axes[1].set_axis_off()

plt.tight_layout()

## Classification

Classify both rasters into discrete zones using quantile breaks. The `quantile` method splits values into equal-frequency bins.

In [ ]:
# Create elevation zones and slope categories using quantile classification
n_elev_classes = 5
elevation_zones = terrain.xrs.quantile(k=n_elev_classes, name='Elevation Zones')

n_slope_classes = 5
slope_classes = slope_agg.xrs.quantile(k=n_slope_classes, name='Slope Classes')

print(f"Created {n_elev_classes} elevation zones and {n_slope_classes} slope classes")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

elevation_zones.plot.imshow(ax=axes[0], cmap='terrain', add_colorbar=True,
                            cbar_kwargs={'label': 'Zone'})
axes[0].set_title('Elevation zones')
axes[0].set_axis_off()

slope_classes.plot.imshow(ax=axes[1], cmap='YlOrRd', add_colorbar=True,
                          cbar_kwargs={'label': 'Class'})
axes[1].set_title('Slope classes')
axes[1].set_axis_off()

plt.tight_layout()

In [ ]:
def bin_ranges(classified_data, original_data, unit="", decimals=0):
    """Calculate the value range for each bin/class."""
    bins = np.unique(classified_data.data[~np.isnan(classified_data.data)])
    ranges = []
    for b in bins:
        bin_data = original_data.data[classified_data.data == b]
        min_val = np.nanmin(bin_data)
        max_val = np.nanmax(bin_data)
        ranges.append(f'{min_val:.{decimals}f}{unit} - {max_val:.{decimals}f}{unit}')
    return ranges

In [ ]:
# Get human-readable labels for each zone
zone_names = ["Valley", "Lowlands", "Foothills", "Mountains", "Peaks"]
slope_names = ["Flat", "Gentle", "Moderate", "Steep", "Very Steep"]

elev_labels = bin_ranges(elevation_zones, terrain, unit='m', decimals=0)
slope_labels = bin_ranges(slope_classes, slope_agg, unit='°', decimals=1)

print("Elevation zones:")
for name, label in zip(zone_names, elev_labels):
    print(f"  {name}: {label}")

print("\nSlope classes:")
for name, label in zip(slope_names, slope_labels):
    print(f"  {name}: {label}")

## Crosstab

Cross-tabulate to see how slope classes are distributed across elevation zones. The `percentage` aggregation shows the fraction of each zone covered by each slope class.

In [ ]:
# Calculate cross-tabulation with percentage aggregation
crosstab_result = slope_classes.xrs.zonal_crosstab(elevation_zones, agg='percentage')

# Add readable labels
crosstab_result['zone'] = [f"{name}\n({label})" for name, label in zip(zone_names, elev_labels)]
crosstab_result.columns = ['Elevation Zone', *[f"{name}\n({label})" for name, label in zip(slope_names, slope_labels)]]
crosstab_result.set_index('Elevation Zone', inplace=True)

crosstab_result

Higher elevation zones tend to have steeper terrain. Valley and lowland areas have more flat and gentle slopes, while mountain and peak zones concentrate in the steep classes.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
crosstab_result.plot(kind="bar", stacked=True, ax=ax, colormap="YlOrRd")
ax.set_xlabel("Elevation Zone")
ax.set_ylabel("Percentage")
ax.set_title("Slope distribution by elevation zone")
ax.legend(title="Slope Class", bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# Save preview image
import pathlib
pathlib.Path('images').mkdir(exist_ok=True)
fig.savefig('images/zonal_crosstab_preview.png', bbox_inches='tight', dpi=120)

## Zonal statistics

For summary statistics (mean, min, max, std) within each zone, use `zonal_stats` instead of `zonal_crosstab`.

In [ ]:
# Mean slope within each elevation zone
mean_slope_by_elev = slope_agg.xrs.zonal_stats(elevation_zones, stats_funcs=['mean', 'std'])
mean_slope_by_elev['Elevation Zone'] = zone_names
mean_slope_by_elev = mean_slope_by_elev[['Elevation Zone', 'mean', 'std']]
mean_slope_by_elev.columns = ['Elevation Zone', 'Mean Slope (°)', 'Std Dev (°)']
mean_slope_by_elev.set_index('Elevation Zone', inplace=True)

mean_slope_by_elev

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
mean_slope_by_elev['Mean Slope (°)'].plot(
    kind="bar", ax=ax, color="orangered",
    yerr=mean_slope_by_elev['Std Dev (°)'], capsize=5)
ax.set_xlabel("Elevation Zone")
ax.set_ylabel("Mean Slope (degrees)")
ax.set_title("Mean terrain steepness by elevation zone")
plt.xticks(rotation=0)
plt.tight_layout()

<div class="alert alert-block alert-info">
<b>Crosstab vs. zonal stats.</b> Use <code>zonal_crosstab</code> when both inputs are categorical (or have been classified into categories). Use <code>zonal_stats</code> when you want summary statistics (mean, std, etc.) of continuous values within each zone. They answer different questions: crosstab shows class distributions, while zonal stats shows value distributions.
</div>

### References

- [Cross-tabulation (Wikipedia)](https://en.wikipedia.org/wiki/Contingency_table)
- [xrspatial.zonal.crosstab API docs](https://xarray-spatial.readthedocs.io/en/latest/reference/_autosummary/xrspatial.zonal.crosstab.html)
- [xrspatial.zonal.stats API docs](https://xarray-spatial.readthedocs.io/en/latest/reference/_autosummary/xrspatial.zonal.stats.html)